# Stage 6: Evaluation & Comparison

- Evaluate all 4 models on the held-out test fold
- Confusion matrices
- Per-class metrics
- Final model comparison bar chart

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import matplotlib.pyplot as plt

from features import load_feature_cache
from evaluate import evaluate_model, plot_comparison
from config   import TEST_FOLD, PLOTS_DIR

plt.rcParams.update({
    'figure.facecolor': '#1a1a2e', 'axes.facecolor': '#0d0d1a',
    'text.color': 'white', 'axes.labelcolor': '#aaa',
    'xtick.color': '#aaa', 'ytick.color': '#aaa',
})

X, y, folds = load_feature_cache('logmel')
print(f'Test fold ({TEST_FOLD}) samples: {(folds == TEST_FOLD).sum()}')

## 1 · Evaluate All Models

In [ ]:
results = []
for model_name in ['rf', 'svm', 'cnn', 'resnet']:
    print(f'\n>>> Evaluating {model_name.upper()} ...')
    try:
        r = evaluate_model(model_name, X, y, folds)
        results.append(r)
    except FileNotFoundError as e:
        print(f'  [SKIP] {e}')

## 2 · Model Comparison

In [ ]:
plot_comparison(results)

import pandas as pd
df_results = pd.DataFrame(results)
df_results = df_results.set_index('model')
df_results[['accuracy', 'precision', 'recall', 'f1']] = \
    df_results[['accuracy', 'precision', 'recall', 'f1']].applymap(lambda x: f'{x:.4f}')
print('\n=== Final Results Table ===')
print(df_results.to_string())

## 3 · Per-class Accuracy Heatmap

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
from evaluate import predict_baseline, predict_dl
from config import IDX_TO_CLASS, NUM_CLASSES

test_mask = folds == TEST_FOLD
X_test, y_test = X[test_mask], y[test_mask]
class_names = [IDX_TO_CLASS[i] for i in range(NUM_CLASSES)]

per_class_accs = {}
for model_name in ['rf', 'svm', 'cnn', 'resnet']:
    try:
        if model_name in ('rf', 'svm'):
            y_pred = predict_baseline(model_name, X_test)
        else:
            y_pred = predict_dl(model_name, X_test)
        cm = confusion_matrix(y_test, y_pred)
        per_class_accs[model_name] = cm.diagonal() / cm.sum(axis=1)
    except:
        pass

if per_class_accs:
    import pandas as pd
    df_pca = pd.DataFrame(per_class_accs, index=class_names)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.heatmap(df_pca * 100, annot=True, fmt='.1f', cmap='YlGnBu',
                ax=ax, linewidths=0.5,
                cbar_kws={'label': 'Accuracy (%)'})
    ax.set_title('Per-class Accuracy by Model', fontsize=13, fontweight='bold', color='white')
    ax.set_xlabel('Model', color='#aaa')
    ax.set_ylabel('Class', color='#aaa')
    plt.tight_layout()
    plt.savefig('../outputs/plots/per_class_accuracy.png', dpi=150, bbox_inches='tight')
    plt.show()